In [ ]:

from google.colab import drive
drive.mount('/content/drive')


data_path = '/content/drive/MyDrive/Amazon_dataset/Electronics_5.json'


Mounted at /content/drive


In [ ]:
!pip install --upgrade transformers
!pip install huggingface_hub[hf_xet]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 106.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.50.3
    Uninstalling transformers-4.50.3:
      Successfully uninstalled transformers-4.50.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 MB 12.6 MB/s eta 0:00:00


In [ ]:
import json
import pandas as pd

json_path = '/content/drive/MyDrive/Amazon_dataset/Electronics_5.json'


def parse_reviews_minimal(json_path):
    data = []

    def convert_score_to_label(score):
        if score >= 4:
            return "positive"
        elif score == 3:
            return "neutral"
        else:
            return "negative"

    with open(json_path, 'r') as f:
        for line in f:
            if line.strip():
                item = json.loads(line.strip())
                sentence = item.get("reviewText", "")
                score = item.get("overall", None)

                if sentence and score is not None:
                    sentiment = convert_score_to_label(score)
                    data.append({"sentence": sentence, "sentiment": sentiment})

    return pd.DataFrame(data)


Amazon_df = parse_reviews_minimal(json_path)
sample_df = Amazon_df.sample(n=50000, random_state=42).reset_index(drop=True)
print(sample_df.head())



                                            sentence sentiment
0  I built a rig for my brother and these fans we...  positive
1  I bought this to copy my old VHS tapes to DVD....  positive
2  EDIT: After about 3 months, the protector is m...  positive
3  Excellent key feedback. Better than some of th...  positive
4  I purchased 3 of these 4 years ago for compute...  positive


In [ ]:
print("Original:", sample_df.shape)

Amazon_dd = sample_df.drop_duplicates()
dd = Amazon_dd.reset_index(drop=True)
print("Drop Duplicates:", dd.shape)

dd_dn = dd.dropna()
df = dd_dn.reset_index(drop=True)
print("Drop Nulls:", df.shape)

Original: (50000, 2)
Drop Duplicates: (49994, 2)
Drop Nulls: (49994, 2)


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r"http\S+|www\S+", '', text)
    text = re.sub(r"[^a-zA-Z0-9\s]", '', text)
    text = re.sub(r"\s+", ' ', text).strip()
    return text

sample_df['sentence'] = sample_df['sentence'].apply(clean_text)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
sample_df['label'] = label_encoder.fit_transform(sample_df['sentiment'])



In [ ]:

from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

encodings = tokenizer.batch_encode_plus(
    sample_df['sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=100,
    return_attention_mask=True,
    return_tensors='pt'
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
sample_df['label'] = label_encoder.fit_transform(sample_df['sentiment'])


tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=100):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
from sklearn.model_selection import train_test_split
def collate_fn(batch):
    texts = [item['text'] if 'text' in item else item.get('raw_text') or "" for item in batch]
    labels = torch.stack([item['label'] for item in batch])

    return {
        'texts': texts,
        'label': labels
    }



train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    sample_df['sentence'],
    sample_df['label'],
    test_size=0.2,
    stratify=sample_df['label'],
    random_state=42
)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.2,
    stratify=train_val_labels,
    random_state=42
)

train_dataset = ReviewDataset(train_texts.tolist(), train_labels.tolist(), tokenizer)
val_dataset = ReviewDataset(val_texts.tolist(), val_labels.tolist(), tokenizer)
test_dataset = ReviewDataset(test_texts.tolist(), test_labels.tolist(), tokenizer)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)



In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

class RobertaBiGRUModel(nn.Module):
    def __init__(self, hidden_size=256, reduced_dim=128, num_classes=3, dropout=0.4, fine_tune_layers=6):
        super(RobertaBiGRUModel, self).__init__()

        self.tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
        self.roberta = RobertaModel.from_pretrained("roberta-base")


        for param in self.roberta.parameters():
            param.requires_grad = True


        if fine_tune_layers > 0:
            for layer in roberta.encoder.layer[-3:]:
                for param in layer.parameters():
                    param.requires_grad = True


        self.bigru = nn.GRU(
            input_size=768,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )


        self.reduce_dim = nn.Sequential(
            nn.Linear(hidden_size * 2, reduced_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )


        self.classifier = nn.Sequential(
            nn.LayerNorm(reduced_dim),
            nn.Linear(reduced_dim, num_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, texts):
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(next(self.parameters()).device)


        with torch.set_grad_enabled(self.roberta.encoder.layer[-1].parameters().__next__().requires_grad):
            outputs = self.roberta(
                input_ids=encoded['input_ids'],
                attention_mask=encoded['attention_mask']
            )
        last_hidden_state = outputs.last_hidden_state


        gru_output, _ = self.bigru(last_hidden_state)
        pooled = torch.mean(gru_output, dim=1)


        reduced = self.reduce_dim(self.dropout(pooled))
        logits = self.classifier(reduced)
        return logits


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.4):
        super(MultiHeadSelfAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.values = nn.Linear(embed_dim, embed_dim, bias=False)
        self.keys = nn.Linear(embed_dim, embed_dim, bias=False)
        self.queries = nn.Linear(embed_dim, embed_dim, bias=False)
        self.fc_out = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        N, seq_len, _ = x.shape


        values = self.values(x)
        keys = self.keys(x)
        queries = self.queries(x)


        values = values.view(N, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(N, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(N, seq_len, self.num_heads, self.head_dim).transpose(1, 2)


        energy = torch.matmul(queries, keys.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-inf"))

        attention = F.softmax(energy, dim=-1)
        attention = self.dropout(attention)

        out = torch.matmul(attention, values)
        out = out.transpose(1, 2).contiguous().view(N, seq_len, self.embed_dim)
        out = self.fc_out(out)
        return self.norm(out + x)


In [ ]:
class DeepFMBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, reduced_dim=128): # Pass reduced_dim as argument
        """
        A simple DeepFM block that models both linear and higher-order interactions.

        Args:
            input_dim (int): Dimension of input features.
            hidden_dim (int): Hidden dimension for the MLP part.
            reduced_dim (int): Dimensionality of the reduced representation.
        """
        super(DeepFMBlock, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.deepfm = nn.Sequential(
              nn.Linear(reduced_dim, 128), # Use reduced_dim here
              nn.BatchNorm1d(128),
              nn.GELU(),
              nn.Dropout(0.4), # Using a common dropout value, change if necessary
              nn.Linear(128, 64),
              nn.BatchNorm1d(64),
              nn.GELU(),
              nn.Dropout(0.4), # Using a common dropout value, change if necessary
              nn.Linear(64, 1)
           )

    def forward(self, x):
        linear_out = self.linear(x)
        deepfm_out = self.deepfm(x) # Changed mlp to deepfm for correctness
        return linear_out + deepfm_out # Changed mlp_out to deepfm_out

In [ ]:
import torch
import torch.nn as nn
from transformers import RobertaTokenizer, RobertaModel

class FullRecommendationModel(nn.Module):
    def __init__(self, hidden_size=256, reduced_dim=128, num_heads=8, num_classes=3, dropout=0.4, fine_tune_layers=4,gru_layers=2):
        super(FullRecommendationModel, self).__init__()

        self.tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
        self.roberta = RobertaModel.from_pretrained('roberta-base')

        for param in self.roberta.parameters():
            param.requires_grad = False
        if fine_tune_layers > 0:
           for layer in self.roberta.encoder.layer[-fine_tune_layers:]:
              for param in layer.parameters():
                  param.requires_grad = True


        self.bigru = nn.GRU(input_size=768,
                            hidden_size=hidden_size,
                            num_layers=1,
                            batch_first=True,
                            bidirectional=True)


        self.dim_reduction = nn.Sequential(
            nn.Linear(hidden_size * 2, reduced_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )


        self.mhsa = nn.MultiheadAttention(embed_dim=reduced_dim, num_heads=num_heads, batch_first=True)


        self.attention_pooling = nn.Sequential(
            nn.Linear(reduced_dim, 1),
            nn.Softmax(dim=1)
        )


        self.deepfm = nn.Sequential(
            nn.Linear(reduced_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )


        self.classifier = nn.Linear(1, num_classes)

        self.dropout = nn.Dropout(dropout)

    def forward(self, texts):

        encoded = self.tokenizer(
                    texts, padding=True, truncation=True, max_length=128, return_tensors='pt'
            ).to(next(self.parameters()).device)


        with torch.set_grad_enabled(any(p.requires_grad for p in self.roberta.encoder.layer[-1].parameters())):
            outputs = self.roberta(
                input_ids=encoded['input_ids'],
                attention_mask=encoded['attention_mask']
            )

        roberta_out = outputs.last_hidden_state


        gru_out, _ = self.bigru(roberta_out)


        reduced = self.dim_reduction(gru_out)


        attn_out, _ = self.mhsa(reduced, reduced, reduced)


        weights = self.attention_pooling(attn_out)
        pooled_output = torch.sum(attn_out * weights, dim=1)


        deepfm_out = self.deepfm(self.dropout(pooled_output))


        logits = self.classifier(deepfm_out)

        return logits

In [ ]:
from tqdm import tqdm



In [ ]:
import torch
from tqdm import tqdm
import torch.nn as nn

def train_model(model, train_loader, val_loader, num_epochs=25, lr=2e-5, patience=4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)

    best_val_loss = float('inf')
    patience_counter = 0

    print("🚀 Starting training...")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for batch in tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] Training"):
            texts = batch['texts']
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        avg_train_loss = total_loss / len(train_loader)
        train_acc = correct / total
        print(f"✅ Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Train Accuracy: {train_acc:.2%}")

        # 🔍 Validation
        val_loss, val_acc = evaluate_model(model, val_loader, device, split="Validation")

        # 🔁 Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print("🔄 Model checkpoint saved.")
        else:
            patience_counter += 1
            print(f"Early stopping trigger: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("🚨 Early stopping triggered!")
                break


In [ ]:
def evaluate_model(model, data_loader, device, split="Test"):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=f"Evaluating on {split} Set"):
            texts = batch['texts']
            labels = batch['label'].to(device)

            outputs = model(texts)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    acc = correct / total
    print(f"📊 {split} Loss: {avg_loss:.4f} | {split} Accuracy: {acc:.2%}")
    return avg_loss, acc


In [ ]:
model = FullRecommendationModel()
train_model(model, train_loader, val_loader, num_epochs=16, lr=2e-5)


evaluate_model(model, test_loader, device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Starting training...


Epoch [1/16] Training: 100%|██████████| 1000/1000 [07:04<00:00,  2.36it/s]


✅ Epoch 1 | Train Loss: 0.5940 | Train Accuracy: 75.98%


Evaluating on Validation Set: 100%|██████████| 250/250 [01:05<00:00,  3.84it/s]


...

Evaluating on Validation Set: 100%|██████████| 250/250 [01:02<00:00,  4.01it/s]


📊 Validation Loss: 0.3844 | Validation Accuracy: 87.29%
Early stopping trigger: 4/4
🚨 Early stopping triggered!


Evaluating on Test Set: 100%|██████████| 313/313 [01:21<00:00,  3.82it/s]

📊 Test Loss: 0.3926 | Test Accuracy: 87.10%


(0.39261584004131367, 0.871)

In [ ]:
torch.save(model.state_dict(), "sentiment_model.pth")
